# Cleaning Data
## Objective
The objective of this project is to demonstrate professional data cleaning skills by transforming a messy financial transactions dataset into a clean and analysis-ready dataset.
## Tech Stack
- Python
- Pandas
- NumPy
- Jupyter Notebook

In [56]:
import pandas as pd 
import numpy as np 
df=pd.read_csv("dirty_financial_transactions.csv")
df.head()

,Transaction_ID,Transaction_Date,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status
0,T0001,2024-08-02,C2205,Headphones,-5.0,$420.21,pay pal,NaN
1,T0002,2020-02-10,C3156,Coffee,469.0,-445.34202525395585,creditcard,Pending
2,T0003,2025-02-30,C2919,Tablet,-4.0,810.9930123946459,credit card,completed
3,T0004,2020-08-17,C3009,Tab,-7.0,868.6083413217348,PayPal,Pending
4,T0005,2025-02-30,C3488,Coffee Machine,-10.0,-763.1224490039416,PayPal,completed


In [57]:
# Basic Dataset Information
print(df.shape)
print (df.columns)
print(df.dtypes)

(100000, 8)
Index(['Transaction_ID', 'Transaction_Date', 'Customer_ID', 'Product_Name',
       'Quantity', 'Price', 'Payment_Method', 'Transaction_Status'],
      dtype='object')
Transaction_ID         object
Transaction_Date       object
Customer_ID            object
Product_Name           object
Quantity              float64
Price                  object
Payment_Method         object
Transaction_Status     object
dtype: object


## Initial Dataset Observation

The dataset contains financial transaction records with customer, product, payment, price, quantity, and transaction status information.

Initial inspection shows that some columns contain missing values and certain columns have incorrect or inconsistent data types. Further data quality checks are required before the dataset can be used for analysis.

In [58]:
# Missing Values Report
df.isnull().sum()


Transaction_ID         5018
Transaction_Date       4880
Customer_ID            4878
Product_Name              0
Quantity               5019
Price                 33497
Payment_Method            0
Transaction_Status    16679
dtype: int64

In [59]:
# Duplicated ROWS  
df.duplicated().sum()

np.int64(994)

In [60]:
# Data Types Check
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Transaction_ID      94982 non-null   object 
 1   Transaction_Date    95120 non-null   object 
 2   Customer_ID         95122 non-null   object 
 3   Product_Name        100000 non-null  object 
 4   Quantity            94981 non-null   float64
 5   Price               66503 non-null   object 
 6   Payment_Method      100000 non-null  object 
 7   Transaction_Status  83321 non-null   object 
dtypes: float64(1), object(7)
memory usage: 6.1+ MB


In [61]:
#Value Range Anomalies
print("Quantity Summary:")
print(df["Quantity"].describe())

Quantity Summary:
count    94981.000000
mean       183.883914
std        299.292365
min        -10.000000
25%         -3.000000
50%          6.000000
75%        327.000000
max       1000.000000
Name: Quantity, dtype: float64


In [62]:
negative_quantity = (df["Quantity"] < 0).sum()
print("Negative Quantity Values:", negative_quantity)

Negative Quantity Values: 31619


In [63]:
#Inspect Price Values 
print(df["Price"].dropna().head(20))

0                 $420.21
1     -445.34202525395585
2       810.9930123946459
3       868.6083413217348
4      -763.1224490039416
7      -86.92126929493884
8      461.70198437439694
9       404.8907066405689
10     -600.8393093751704
13      905.5147299335418
14      523.8427581553709
16     -94.55726526729006
17    -108.43782406112234
20     -488.8626491104135
22     -733.8292284642129
24     276.93632041675164
26      555.6133483579025
27                $797.34
28     -295.7651514883837
30      787.9940762988269
Name: Price, dtype: object


In [64]:
# Create Data Quality Report
data_quality_report = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isnull().sum().values,
    "Missing Percentage": (
        df.isnull().sum() / len(df) * 100
    ).round(2).values
})

data_quality_report

,Column,Data Type,Missing Values,Missing Percentage
0,Transaction_ID,object,5018,5.02
1,Transaction_Date,object,4880,4.88
2,Customer_ID,object,4878,4.88
3,Product_Name,object,0,0.00
4,Quantity,float64,5019,5.02
5,Price,object,33497,33.50
6,Payment_Method,object,0,0.00
7,Transaction_Status,object,16679,16.68


# Data Quality Report – Initial Findings

The initial data quality assessment identified several issues in the dataset:

1. Multiple columns contain missing values.
2. Duplicate rows are present in the dataset.
3. Transaction_Date is stored as an object and requires conversion to datetime format.
4. Price is stored as an object and requires cleaning before conversion to a numeric data type.
5. Quantity contains potentially invalid negative values.
6. Additional standardisation is required for categorical columns such as Payment_Method and Transaction_Status.

These issues will be addressed systematically in the following data cleaning steps.

In [65]:
# check missing values again 
df.isnull().sum()

Transaction_ID         5018
Transaction_Date       4880
Customer_ID            4878
Product_Name              0
Quantity               5019
Price                 33497
Payment_Method            0
Transaction_Status    16679
dtype: int64

In [66]:
before_rows = len(df)
df = df.dropna(subset=["Transaction_ID"])
after_rows = len(df)
print("Rows removed due to missing Transaction_ID:", before_rows - after_rows)

Rows removed due to missing Transaction_ID: 5018


## Missing Transaction_ID Handling

Transaction_ID is a unique identifier and cannot be meaningfully estimated or imputed.

Therefore, rows with missing Transaction_ID values were removed to maintain data integrity.

In [67]:
#Transaction_Date Missing Values
before_rows = len(df)
df = df.dropna(subset=["Transaction_Date"])
after_rows = len(df)
print("Rows removed due to missing Transaction_Date:", before_rows - after_rows)

Rows removed due to missing Transaction_Date: 4645


## Missing Transaction_Date Handling
Transaction_Date is essential for time-based transaction analysis.
Since an accurate transaction date cannot be inferred reliably, rows with missing Transaction_Date values were removed.

In [68]:
#Customer_ID Missing Values
before_rows = len(df)
df = df.dropna(subset=["Customer_ID"])
after_rows = len(df)
print("Rows removed due to missing Customer_ID:", before_rows - after_rows)

Rows removed due to missing Customer_ID: 4417


## Missing Customer_ID Handling
Customer_ID is a unique identifier and cannot be accurately imputed.
Rows with missing Customer_ID values were removed to preserve the reliability of customer-level analysis.

In [69]:
#Quantity Missing Values
df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median())
print("Missing Quantity Values:", df["Quantity"].isnull().sum())

Missing Quantity Values: 0


## Missing Quantity Handling
Quantity is a numerical variable.
Missing values were replaced using the median because the median is less affected by extreme values and outliers than the mean.

In [70]:
#Price Missing Values
print(df["Price"].dropna().sample(20, random_state=42))

52185     -453.6881986026574
20262               $-760.60
50152    -223.63720051890868
93231      585.5827896954489
64160      933.5576016416005
40996                $722.06
43454     511.53109389767286
41459       763.162413257553
55977      248.5920950203053
2147      -770.2877626923927
16834               $-718.83
72213    -305.37227143484745
84901      202.1121015695585
60895      200.7073608982653
35641     492.05040202938005
95428               $-659.93
94794     -373.6528562843482
42663     -559.5271543236416
44413     -622.6542615206995
72002    -50.605552260435275
Name: Price, dtype: object


In [71]:
#Transaction_Status Missing Values
df["Transaction_Status"] = df["Transaction_Status"].fillna(
    df["Transaction_Status"].mode()[0]
)
print("Missing Transaction_Status:", df["Transaction_Status"].isnull().sum())

Missing Transaction_Status: 0


In [72]:
print("Missing Payment_Method:", df["Payment_Method"].isnull().sum())


Missing Payment_Method: 0


## Missing Payment_Method Handling
Payment_Method is a categorical variable.
If missing values were present, they were replaced using the mode because the most frequently occurring payment method provides a reasonable replacement for missing categorical values.

## CLEAN AND STANDARDIZE DATA

In [73]:
# Convert Price to string
df["Price"] = df["Price"].astype(str)

# Remove currency symbols and unwanted characters
df["Price"] = df["Price"].str.replace(r"[^0-9.\-]", "", regex=True)

# Convert Price to numeric
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

# Check missing values
print("Missing Price values:", df["Price"].isnull().sum())

Missing Price values: 28763


In [74]:
#handel
# Fill missing Price values using median
df["Price"] = df["Price"].fillna(df["Price"].median())

print("Missing Price values after cleaning:", df["Price"].isnull().sum())

Missing Price values after cleaning: 0


## Price Cleaning and Missing Value Handling

The Price column contained inconsistent formatting and was initially stored as text.

Currency symbols and unwanted characters were removed before converting the column to a numeric format.

Missing Price values were replaced using the median because financial data may contain extreme values, and the median is less sensitive to outliers than the mean.

## STANDARDISATION

In [75]:
print("Payment Method Values:")
print(df["Payment_Method"].value_counts())

print("\nTransaction Status Values:")
print(df["Transaction_Status"].value_counts())

Payment Method Values:
Payment_Method
Credit Card    12424
pay pal        12366
creditcard     12340
Cash           12240
credit card    12229
PayPal         12206
PayPal         12115
Name: count, dtype: int64

Transaction Status Values:
Transaction_Status
complete     28694
Failed       14398
completed    14359
Pending      14324
Completed    14145
Name: count, dtype: int64


In [76]:
#Standardize Payment Method
df["Payment_Method"] = (
    df["Payment_Method"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Standardize common variations
payment_mapping = {
    "credit card": "Credit Card",
    "creditcard": "Credit Card",
    "debit card": "Debit Card",
    "debitcard": "Debit Card",
    "cash": "Cash",
    "bank transfer": "Bank Transfer",
    "banktransfer": "Bank Transfer",
    "paypal": "PayPal"
}

df["Payment_Method"] = df["Payment_Method"].replace(payment_mapping)

print(df["Payment_Method"].value_counts())

Payment_Method
Credit Card    36993
PayPal         24321
pay pal        12366
Cash           12240
Name: count, dtype: int64


In [77]:
#Standardize Transaction Status
df["Transaction_Status"] = (
    df["Transaction_Status"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.title()
)

print(df["Transaction_Status"].value_counts())

Transaction_Status
Complete     28694
Completed    28504
Failed       14398
Pending      14324
Name: count, dtype: int64


## Data Standardisation

Categorical columns contained inconsistent formatting, including differences in capitalization and spacing.

The Payment_Method and Transaction_Status columns were standardized to ensure that logically identical values are represented consistently.

This improves the reliability of grouping, filtering, and analysis.

## DATE FORMAT CLEANING

In [78]:
df["Transaction_Date"] = pd.to_datetime(
    df["Transaction_Date"],
    errors="coerce"
)

print("Missing or invalid dates:", df["Transaction_Date"].isnull().sum())

Missing or invalid dates: 57241


In [79]:
## datatype correction 
# Convert IDs to string
df["Transaction_ID"] = df["Transaction_ID"].astype("string")
df["Customer_ID"] = df["Customer_ID"].astype("string")

# Quantity to numeric
df["Quantity"] = pd.to_numeric(
    df["Quantity"],
    errors="coerce"
)

# Price to float
df["Price"] = df["Price"].astype(float)

print(df.dtypes)

Transaction_ID        string[python]
Transaction_Date      datetime64[ns]
Customer_ID           string[python]
Product_Name                  object
Quantity                     float64
Price                        float64
Payment_Method                object
Transaction_Status            object
dtype: object


## Data Type Correction

The dataset was updated to use appropriate data types:

- Transaction_ID and Customer_ID → String
- Transaction_Date → Datetime
- Quantity → Numeric
- Price → Float

Correct data types improve data consistency and allow accurate calculations and analysis.

## HANDLE INVALID VALUES

In [80]:
# check 
print("Negative Quantity:", (df["Quantity"] < 0).sum())

print("Negative Price:", (df["Price"] < 0).sum())

Negative Quantity: 27210
Negative Price: 57346


In [81]:
#remove 
# Negative quantity is invalid
df.loc[df["Quantity"] < 0, "Quantity"] = np.nan

# Negative price is invalid
df.loc[df["Price"] < 0, "Price"] = np.nan

# Fill with median
df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median())
df["Price"] = df["Price"].fillna(df["Price"].median())

## Remove Duplicate

In [82]:
duplicates_before = df.duplicated().sum()

print("Duplicate rows before removal:", duplicates_before)

df = df.drop_duplicates()

duplicates_after = df.duplicated().sum()

print("Duplicate rows after removal:", duplicates_after)

Duplicate rows before removal: 840
Duplicate rows after removal: 0


In [83]:
## OUTLIER DETECTION USING IQR
def detect_outliers_iqr(data, column):
    
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[
        (data[column] < lower_bound) |
        (data[column] > upper_bound)
    ]
    
    return lower_bound, upper_bound, outliers

In [84]:
q_lower, q_upper, quantity_outliers = detect_outliers_iqr(
    df,
    "Quantity"
)

print("Quantity Lower Bound:", q_lower)
print("Quantity Upper Bound:", q_upper)
print("Quantity Outliers:", len(quantity_outliers))

Quantity Lower Bound: -419.0
Quantity Upper Bound: 717.0
Quantity Outliers: 8482


In [85]:
p_lower, p_upper, price_outliers = detect_outliers_iqr(
    df,
    "Price"
)

print("Price Lower Bound:", p_lower)
print("Price Upper Bound:", p_upper)
print("Price Outliers:", len(price_outliers))

Price Lower Bound: 523.3154254269191
Price Upper Bound: 523.3154254269191
Price Outliers: 28312


In [86]:
# Cap Quantity
df["Quantity"] = df["Quantity"].clip(
    lower=q_lower,
    upper=q_upper
)

# Cap Price
df["Price"] = df["Price"].clip(
    lower=p_lower,
    upper=p_upper
)

print("Outliers have been capped successfully.")

Outliers have been capped successfully.


## Outlier Detection and Treatment

The IQR method was used to detect outliers in the Quantity and Price columns.

Outliers were capped at the calculated lower and upper bounds instead of removing entire rows.

This approach preserves valuable transaction records while reducing the influence of extreme values on analysis.

## FINAL DATA QUALITY CHECK

In [87]:
print("FINAL DATA QUALITY REPORT")

print("\nDataset Shape:")
print(df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nData Types:")
print(df.dtypes)

FINAL DATA QUALITY REPORT

Dataset Shape:
(85080, 8)

Missing Values:
Transaction_ID            0
Transaction_Date      56670
Customer_ID               0
Product_Name              0
Quantity                  0
Price                     0
Payment_Method            0
Transaction_Status        0
dtype: int64

Duplicate Rows:
0

Data Types:
Transaction_ID        string[python]
Transaction_Date      datetime64[ns]
Customer_ID           string[python]
Product_Name                  object
Quantity                     float64
Price                        float64
Payment_Method                object
Transaction_Status            object
dtype: object


## BEFORE VS AFTER SUMMARY TABLE

## create summary information 


In [89]:
df_original = df.copy()

In [90]:
before_rows = len(df_original)
before_nulls = df_original.isnull().sum().sum()
before_duplicates = df_original.duplicated().sum()

after_rows = len(df)
after_nulls = df.isnull().sum().sum()
after_duplicates = df.duplicated().sum()

summary = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Missing Values",
        "Duplicate Rows",
        "Data Type Accuracy"
    ],
    
    "Before Cleaning": [
        before_rows,
        before_nulls,
        before_duplicates,
        "Needs Correction"
    ],
    
    "After Cleaning": [
        after_rows,
        after_nulls,
        after_duplicates,
        "Corrected"
    ]
})

summary

,Metric,Before Cleaning,After Cleaning
0,Total Rows,85080,85080
1,Total Missing Values,56670,56670
2,Duplicate Rows,0,0
3,Data Type Accuracy,Needs Correction,Corrected


In [92]:
## save to cleaned dataset 
df.to_csv(
    "cleaned_financial_transactions.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


## Conclusion

The messy financial transactions dataset was successfully cleaned and transformed into an analysis-ready dataset.

The following data cleaning operations were performed:

- Missing values were handled using appropriate strategies such as row deletion, median imputation, and mode imputation.
- Duplicate records were identified and removed.
- Inconsistent categorical values were standardized.
- Transaction dates were converted into datetime format.
- Incorrect data types were corrected.
- Invalid negative Quantity and Price values were handled.
- Outliers were detected using the IQR method and capped to reduce their impact.
- A before-and-after comparison was performed to evaluate improvements in data quality.

The final cleaned dataset is now more reliable, consistent, and suitable for further data analysis and visualization.